In [20]:
# import cpuinfo

# cpu_name = cpuinfo.get_cpu_info()['brand_raw']
# print(f"CPU Name: {cpu_name}")

import torch

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA GPU found.")



GPU Name: NVIDIA GeForce RTX 5060 Laptop GPU


# Autograd

When neural networks get too deep, it is impossible to calculate derivatives for backpropagation. Autograd helps us with this problem by automatically calculating the derivatives for us. It does this by keeping track of all the operations that we perform on tensors, and then using the chain rule to calculate the derivatives when we call `backward()`.

In [11]:
x = torch.tensor(1.0, requires_grad=True)

print(x)

y = x**2

print(y)

tensor(1., requires_grad=True)
tensor(1., grad_fn=<PowBackward0>)


Now, look at the `grad_fn=<PowBackward0>` in the output. This means that `y` is a result of an operation (in this case, the power operation) that has a gradient function associated with it. This is how PyTorch keeps track of the operations that we perform on tensors, and it allows us to calculate the derivatives when we call `backward()`.

In [12]:
y.backward()

x.grad

print(x.grad)

tensor(2.)


Now let's calculate $\frac{dz}{dx}$ where $y = x^2$ and $z = \sin y$

In [13]:
x = torch.tensor(3.0, requires_grad=True)

y = x**2

z = torch.sin(y)

print(x, y, z)

z.backward()

print(x.grad)

import math

print(2 * 3 * math.cos(3**2) )

# it is the same!!

tensor(3., requires_grad=True) tensor(9., grad_fn=<PowBackward0>) tensor(0.4121, grad_fn=<SinBackward0>)
tensor(-5.4668)
-5.466781571308061


Note that since $y$ is a "leaf node", `y.grad` will return an error. Only tensors that are created as a result of an operation (i.e., non-leaf nodes) will have a `grad_fn` and will be able to calculate gradients when we call `backward()`.

We _can_ generate leaf gradients, but that's later.

## Another example

1. Linear Transformation: $z = wx + b$, where $w$ is the weight, $x$ is the input, and $b$ is the bias.
2. Activation Function: $\hat{y} = \sigma(z)$, where $\sigma$ is the sigmoid activation function. $\sigma(z) = \frac{1}{1 + e^{-z}}$
3. Loss Function (binary cross entropy loss): $L = -\frac{1}{N}\sum_{i=1}^{N}[t_i \log(\hat{y}_i) + (1-t_i) \log(1-\hat{y}_i)]$, where $t_i$ are the target output and $N$ is the number of samples.

For $1$ sample, the loss function simplifies to: $L = -[t \log(\hat{y}) + (1-t) \log(1-\hat{y})]$

Now, $\frac{\partial L}{\partial w}$ can be calculated using the chain rule as follows:
$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} \cdot \frac{\partial z}{\partial w}$$
And, $$\frac{\partial L}{\partial b} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} \cdot \frac{\partial z}{\partial b}$$


Now, $$\frac{\partial L}{\partial \hat{y}} = \frac{\hat{y} - t}{\hat{y}(1-\hat{y})}$$
$$\frac{\partial \hat{y}}{\partial z} = \hat{y}(1-\hat{y})$$
$$\frac{\partial z}{\partial w} = x$$
$$\frac{\partial z}{\partial b} = 1$$

### Then,
$$\boxed{\frac{\partial L}{\partial w} = \frac{\hat{y} - t}{\hat{y}(1-\hat{y})} \cdot \hat{y}(1-\hat{y}) \cdot x = (\hat{y} - t) \cdot x}$$
$$\boxed{\frac{\partial L}{\partial b} = \frac{\hat{y} - t}{\hat{y}(1-\hat{y})} \cdot \hat{y}(1-\hat{y}) \cdot 1 = (\hat{y} - t)}$$


In [14]:
# Now to implement the code

x = torch.tensor(6.7) # input 

y = torch.tensor(0.0) # target

w = torch.tensor(1.0, requires_grad=True) # weight
b = torch.tensor(0.0, requires_grad=True) # bias

In [15]:
def binary_cross_entropy_loss(y_pred, target):
    
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)
    # to avoid log(0) which is undefined
    
    
    return - (target * torch.log(y_pred) + (1 - target) * torch.log(1 - y_pred)).mean()

In [16]:
# forward pass:

z = w * x + b

y_pred = torch.sigmoid(z)


# calculate loss

loss = binary_cross_entropy_loss(y_pred, y) # putting y as target

In [ ]:
# backward pass:
loss.backward() # compute gradients

print(w.grad) # d(loss)/d(w)
print(b.grad) # d(loss)/d(b)

tensor(6.6918)
tensor(0.9988)


In [21]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

y = (x ** 2).mean() # y = f(x1, x2, x3) = (x1^2 + x2^2 + x3^2)/3


y.backward() # compute gradients

print(x.grad) # [d(y)/d(x1), d(y)/d(x2), d(y)/d(x3)] -> [0.6667, 1.3333, 2.0000]

tensor([0.6667, 1.3333, 2.0000])


Remember: if we do repeated forward and backward passes, the gradients will accumulate in the `.grad` attribute. To prevent this, we can zero out the gradients before each backward pass using `x.grad.zero_()`.